In [4]:
import polars as pl
from pathlib import Path
import scipy


BASE_PATH = Path.cwd().parent
SOURCE_DATA_PATH = BASE_PATH.joinpath("part_i", "metric_comparison_source_data_cleaned.csv")


In [40]:
df = pl.read_csv(SOURCE_DATA_PATH)

In [41]:
df

nro_pregunta,nro_subpregunta,estrategia,id_modelo_especialista,metrica,valor
i64,i64,str,str,str,f64
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""exact_match""",0.0
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""token_f1""",0.1976
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""jaccard_similarity""",0.1096
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""levenshtein_similarity""",0.1734
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""character_error_rate""",3.8154
…,…,…,…,…,…
25,null,"""llm_responses_agentic_rag""","""gpt-5-mini""","""sbert_similarity_multilingual""",0.7897
25,null,"""llm_responses_agentic_rag""","""gpt-5-mini""","""sbert_similarity_spanish""",0.6681
25,null,"""llm_responses_agentic_rag""","""gpt-5-mini""","""bertscore""",0.6943


TypeError: DataFrame.min() takes 1 positional argument but 2 were given

In [38]:
df.filter(pl.col("metrica") == "jaccard_similarity")

nro_pregunta,nro_subpregunta,estrategia,id_modelo_especialista,metrica,valor
i64,i64,str,str,str,f64
1,1,"""llm_responses_no_rag""","""gpt-5-nano""","""jaccard_similarity""",0.1096
1,1,"""llm_responses_no_rag""","""gpt-5-mini""","""jaccard_similarity""",0.0762
1,1,"""llm_responses_no_rag""","""gpt-5""","""jaccard_similarity""",0.1525
1,1,"""llm_responses_base_rag""","""gpt-5-nano""","""jaccard_similarity""",0.1125
1,1,"""llm_responses_base_rag""","""gpt-5-mini""","""jaccard_similarity""",0.1234
…,…,…,…,…,…
25,null,"""llm_responses_base_rag""","""gpt-5-mini""","""jaccard_similarity""",0.1944
25,null,"""llm_responses_base_rag""","""gpt-5""","""jaccard_similarity""",0.2444
25,null,"""llm_responses_agentic_rag""","""gpt-5""","""jaccard_similarity""",0.15


In [59]:
df.select(
    pl.col("estrategia"),
    pl.col("id_modelo_especialista"),
    pl.col("metrica"),
    pl.col("valor"),
).filter(
    pl.col("metrica") == "jaccard_similarity"
).group_by(
    pl.col("estrategia"),
    pl.col("id_modelo_especialista"),
).agg(
    pl.col("valor").min().alias("min_jaccard_similarity"),
    pl.col("valor").max().alias("max_jaccard_similarity"),
    pl.col("valor").mean().alias("mean_jaccard_similarity"),
    pl.col("valor").std().alias("std_jaccard_similarity"),
    pl.col("valor").len().alias("count_jaccard_similarity"),
).sort(
    pl.col("id_modelo_especialista"),
    pl.col("estrategia"),
)

estrategia,id_modelo_especialista,min_jaccard_similarity,max_jaccard_similarity,mean_jaccard_similarity,std_jaccard_similarity,count_jaccard_similarity
str,str,f64,f64,f64,f64,u32
"""llm_responses_agentic_rag""","""gpt-5""",0.0872,0.4245,0.177011,0.060395,79
"""llm_responses_base_rag""","""gpt-5""",0.0634,0.4359,0.171137,0.071981,79
"""llm_responses_no_rag""","""gpt-5""",0.0553,0.2752,0.138252,0.047398,79
"""llm_responses_agentic_rag""","""gpt-5-mini""",0.0492,0.3333,0.127961,0.047768,79
"""llm_responses_base_rag""","""gpt-5-mini""",0.0413,0.3684,0.135404,0.062242,79
"""llm_responses_no_rag""","""gpt-5-mini""",0.0393,0.1902,0.101177,0.028953,79
"""llm_responses_agentic_rag""","""gpt-5-nano""",0.0,0.3483,0.091761,0.098315,79
"""llm_responses_base_rag""","""gpt-5-nano""",0.0571,0.4588,0.159905,0.084988,79
"""llm_responses_no_rag""","""gpt-5-nano""",0.0558,0.2194,0.11488,0.037963,79


['bleu',
 'jaccard_similarity',
 'bertscore',
 'levenshtein_similarity',
 'sbert_similarity_spanish',
 'rouge_2',
 'sbert_similarity_multilingual',
 'character_error_rate',
 'word_count_ratio',
 'length_ratio',
 'rouge_l',
 'token_f1',
 'rouge_1',
 'exact_match']

In [79]:
from plotnine import ggplot, aes, geom_boxplot, labs, theme_minimal, geom_jitter, theme, element_text

PLOTS_PATH = BASE_PATH.joinpath("part_i", "plots")
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

for metrica in list(df.unique(pl.col("metrica"))["metrica"]):
    plot = (
        ggplot(
            df.filter(pl.col("metrica") == metrica),
            aes(x="estrategia", y="valor", color="id_modelo_especialista"),
        )
        + geom_boxplot()
        + geom_jitter(width=0.2, alpha=0.3)
        + labs(title=f"{metrica} by Strategy", x="Strategy", y="Value", color="Model")
        + theme_minimal()
        + theme(legend_position="bottom", legend_direction="horizontal")
    )

    plot.save(PLOTS_PATH.joinpath(f"{metrica}_boxplot.png"), dpi=300, width=10, height=6)

/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 10 x 6 in image.
/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /mnt/c/Users/leo24/OneDrive/ex_dropbox/tesis/examples/notebooks/part_i/plots/length_ratio_boxplot.png
/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 10 x 6 in image.
/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /mnt/c/Users/leo24/OneDrive/ex_dropbox/tesis/examples/notebooks/part_i/plots/character_error_rate_boxplot.png
/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 10 x 6 in image.
/home/leospairani/.uv-envs/tesis-notebooks/lib/python3.13/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /mnt/c/U